# Building ML APIs with FastAPI

## 🎯 Learning Objectives

- Build REST APIs for ML models
- Handle request validation
- Create API documentation
- Implement async endpoints
- Add authentication and error handling

## Why FastAPI?

- **Fast**: High performance (comparable to NodeJS/Go)
- **Easy**: Intuitive Python syntax
- **Auto-docs**: Swagger UI out of the box
- **Type safety**: Pydantic validation
- **Async**: Native async/await support


In [ ]:
# Install dependencies
# !pip install fastapi uvicorn pydantic


## Hello World API


In [ ]:
from fastapi import FastAPI

app = FastAPI(title="My First ML API")

@app.get("/")
async def root():
    return {"message": "Hello, MLOps!"}

@app.get("/health")
async def health_check():
    return {"status": "healthy"}

print("API created!")
print("Run with: uvicorn main:app --reload")


## Request/Response Models with Pydantic


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class TextInput(BaseModel):
    text: str = Field(..., min_length=1, max_length=1000)
    language: str = Field(default="en")

class SentimentOutput(BaseModel):
    label: str
    confidence: float
    scores: dict

# Example usage
sample_input = TextInput(
    text="FastAPI is amazing!",
    language="en"
)

print(sample_input.model_dump())


## ML Model Endpoint


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import joblib

app = FastAPI(title="Sentiment Analysis API")

# Mock training data
train_texts = [
    "I love this product",
    "This is amazing",
    "Terrible experience",
    "Very disappointing"
]
train_labels = [1, 1, 0, 0]  # 1=positive, 0=negative

# Train simple model
vectorizer = TfidfVectorizer(max_features=100)
X_train = vectorizer.fit_transform(train_texts)
model = MultinomialNB()
model.fit(X_train, train_labels)

print("✓ Model trained")

class TextRequest(BaseModel):
    text: str

class SentimentResponse(BaseModel):
    sentiment: str
    confidence: float

@app.post("/predict", response_model=SentimentResponse)
async def predict_sentiment(request: TextRequest):
    try:
        # Vectorize input
        X = vectorizer.transform([request.text])
        
        # Predict
        prediction = model.predict(X)[0]
        probabilities = model.predict_proba(X)[0]
        
        sentiment = "positive" if prediction == 1 else "negative"
        confidence = float(probabilities[prediction])
        
        return SentimentResponse(
            sentiment=sentiment,
            confidence=confidence
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✓ Sentiment endpoint created")


## Batch Predictions


In [ ]:
from typing import List

class BatchRequest(BaseModel):
    texts: List[str]

class BatchResponse(BaseModel):
    predictions: List[SentimentResponse]

@app.post("/batch_predict", response_model=BatchResponse)
async def batch_predict(request: BatchRequest):
    predictions = []
    
    for text in request.texts:
        X = vectorizer.transform([text])
        pred = model.predict(X)[0]
        proba = model.predict_proba(X)[0]
        
        predictions.append(SentimentResponse(
            sentiment="positive" if pred == 1 else "negative",
            confidence=float(proba[pred])
        ))
    
    return BatchResponse(predictions=predictions)

print("✓ Batch endpoint created")


## Error Handling


In [ ]:
from fastapi import HTTPException, status

@app.post("/predict_with_validation")
async def predict_with_validation(request: TextRequest):
    # Validate input
    if len(request.text) < 3:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Text must be at least 3 characters"
        )
    
    if len(request.text) > 1000:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Text too long (max 1000 characters)"
        )
    
    # Process request
    X = vectorizer.transform([request.text])
    prediction = model.predict(X)[0]
    
    return {
        "sentiment": "positive" if prediction == 1 else "negative"
    }


## API Documentation

FastAPI automatically generates interactive documentation:

- **Swagger UI**: http://localhost:8000/docs
- **ReDoc**: http://localhost:8000/redoc

No extra work needed!


## Running the API

```bash
# Development mode (auto-reload)
uvicorn main:app --reload

# Production mode
uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4
```


## Testing the API

```bash
# Using curl
curl -X POST "http://localhost:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{"text": "This is fantastic!"}'

# Using Python
import requests

response = requests.post(
    "http://localhost:8000/predict",
    json={"text": "This is fantastic!"}
)

print(response.json())
```


## Best Practices

1. **Use Pydantic models** for request/response validation
2. **Add proper error handling** with HTTPException
3. **Implement health checks** for monitoring
4. **Use async/await** for I/O operations
5. **Add API versioning** (/v1/predict, /v2/predict)
6. **Document endpoints** with docstrings
7. **Add rate limiting** for production
8. **Implement authentication** for security

## Key Takeaways

✅ FastAPI makes building ML APIs simple
✅ Pydantic ensures type safety and validation
✅ Auto-generated docs save time
✅ Async support for better performance
✅ Ready for production with proper error handling
